In [ ]:
import os
from datasets import load_dataset
import os
from tqdm import tqdm

base_path = os.getcwd()
data_path = os.path.join(base_path, 'wav')

data = load_dataset("audiofolder", data_dir = data_path)
data

In [ ]:
 # tqdm is used for the progress bar
# Assuming 'data' is your dataset loaded with Hugging Face datasets
audio_filenames = []

# Loop through the dataset with a progress bar
for item in tqdm(data['train'], desc="Extracting file names"):
    audio_path = item['audio']['path']
    audio_filename = os.path.basename(audio_path)  # Extract the file name
    audio_name = os.path.splitext(audio_filename)[0]  # Remove the .wav extension
    audio_filenames.append(audio_name)


In [ ]:

def extract_transcripts(audio_filenames, transcript_file, filename_output, text_output):
    # Create a dictionary to hold audio filename to text mappings
    audio_to_text = {}

    # Read the transcript from the specified file
    with open(transcript_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    # Process each line in the transcript
    for line in lines:
        # Split the line into filename and text
        parts = line.strip().split(' ', 1)  # Split into two parts: filename and text
        if len(parts) == 2:
            filename, text = parts
            # Remove spaces from the text
            text_no_spaces = text.replace(' ', '')
            # If the filename (without .wav) is in the audio filenames list, map it to the text
            if filename in audio_filenames:
                audio_to_text[filename] = text_no_spaces

    # Write the extracted filenames to the filename output file
    with open(filename_output, 'w', encoding='utf-8') as fname_file:
        for filename in audio_to_text.keys():
            fname_file.write(f"{filename}.wav\n")

    # Write the extracted texts to the text output file
    with open(text_output, 'w', encoding='utf-8') as text_file:
        for text in audio_to_text.values():
            text_file.write(f"{text}\n")
    
    return audio_to_text

# Example usage
# Assuming 'audio_filenames' is a list of audio file names without extensions
transcript_file = os.path.join(base_path, 'transcript', 'transcript.txt')  # Path to your transcript file
output_file = 'filename_withTranscript.txt'  # Path to the output file

# Call the function
result = extract_transcripts(audio_filenames, transcript_file, output_file, 'checked_transcripts.txt')

#### Filtering the dataset with correct audio and transcriptions

In [ ]:

def filter_data_based_on_transcripts(data, transcript_file):
    # Read all lines from the transcript file and strip whitespace
    with open(transcript_file, 'r', encoding='utf-8') as file:
        lines = [line.strip() for line in file.readlines()]
    
    # Initialize a list to store indices of items to keep
    indices_to_keep = []
    
    # Loop through each item in the dataset
    for idx, item in tqdm(enumerate(data['train']), desc="Filtering dataset"):
        audio_path = item['audio']['path']
        audio_filename = os.path.basename(audio_path)  # Extract the file name
        
        # If the filename is in the transcript file, keep the index
        if audio_filename in lines:
            indices_to_keep.append(idx)
    
    # Filter the dataset to keep only the indices we want
    filtered_data = data['train'].select(indices_to_keep)
    
    return filtered_data

# Example usage
transcript_file = 'filename_withTranscript.txt'  # The path to your transcript file
data['train'] = filter_data_based_on_transcripts(data, transcript_file)
    

### Transcript conversion and insertion --> OpenSlr33 

In [ ]:
# Step 1: Read the cleaned sentences from the 'cleaned_transcript.txt' file
sentences = []
with open('checked_transcripts.txt', 'r', encoding='utf-8') as f:
    sentences = f.readlines()
    
# Step 2: Clean up the sentence list (strip newline characters)
sentences = [sentence.strip() for sentence in sentences]

# Step 3: Add the 'sentence' feature to the existing dataset
# We can use the `map` function to add the sentences to each row in the dataset
def add_sentences(example, idx):
    example['sentence'] = sentences[idx]
    return example

# Apply the 'add_sentences' function to each row of the dataset
updated_dataset = data['train'].map(add_sentences, with_indices=True)

# Update the dataset
data['train'] = updated_dataset

data

### Add transcript into the dataset

In [ ]:
import re
from dragonmapper import hanzi

def remove_tones(ipa_string):
    # Define a pattern to match only tone markers (˥, ˧, ˩, etc.)
    pattern = r'[˥˧˩˦˨×]+'
    
    # Remove the tone markers using re.sub()
    ipa_string_no_tones = re.sub(pattern, '', ipa_string)
    
    return ipa_string_no_tones

def remove_punctuation(input_string):
    # Updated regex pattern to include a wide range of punctuation marks and spaces
    pattern = r"[!\"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~·•《》「」『』【】…（）、；：！？——‘’“”，‧。“”·：、/ㄟＰ|・／－〉〈─□Λ]+"
    
    # Remove the unwanted characters
    cleaned_string = re.sub(pattern, '', input_string)
    
    return cleaned_string

# List of exceptions to keep together
exceptions = ['pʰ', 'ts', 'tsʰ', 'tʰ', 'ʈʂ', 'ʈʂʰ', 'tɕ', 'tɕʰ', 'kʰ', 'ɑɻ', 'ai', 'ei', 'ɑʊ', 'oʊ', 'ia', 'iɛ', 'wa', 'wɔ', 'ɥœ', 'iɑʊ', 'ioʊ', 'wai', 'weɪ']
def convert_phoneme(input_string):
    try:
        # Remove punctuation
        input_string = remove_punctuation(input_string)
        
        # Convert to IPA
        ipa_result = hanzi.to_ipa(input_string, delimiter=' ', all_readings=False, container='[]')
        ipa_result = ipa_result.replace('j', 'i').replace('ɪ', 'i')
        
        # Remove tones and spaces from the IPA result
        ipa_result = remove_tones(ipa_result)
        ipa_result = ' '.join(ipa_result.split())  # Removing all spaces
        
        symbols = list(ipa_result)
        result = []
        i = 0
        
        while i < len(symbols):
            
            # Check for syllable 'ɻ' and replace it with 'ɑɻ'
            if symbols[i] == 'ɻ':
                result.append('ɑɻ')
                i += 1  # Move to the next character
                
            # Try matching 3-character exceptions first
            elif i < len(symbols) - 2 and (symbols[i] + symbols[i + 1] + symbols[i + 2]) in exceptions:
                result.append(symbols[i] + symbols[i + 1] + symbols[i + 2])
                i += 3  # Skip the next two characters since we've processed them as part of the exception
            # Try matching 2-character exceptions
            elif i < len(symbols) - 1 and (symbols[i] + symbols[i + 1]) in exceptions:
                result.append(symbols[i] + symbols[i + 1])
                i += 2  # Skip the next character since we've processed it as part of the exception
            else:
                # No match, so treat the current character as a separate symbol
                result.append(symbols[i])
                i += 1
        
        return ' '.join(result)

    except ValueError as e:
        print(f"Error processing syllable: {input_string}")
        print(f"Error details: {e}")
        raise


# Define a function to add the 'transcript' column
def add_transcript_column(example):
    example['transcript'] = convert_phoneme(example['sentence'])
    return example


# Apply this function to add 'transcript' to the 'train', 'validation', and 'test' datasets
data['train'] = data['train'].map(add_transcript_column)

#### Split the dataset

In [ ]:
from datasets import load_dataset, DatasetDict

# Step 1: Split the first 70% for training
train_test_split = data['train'].train_test_split(test_size=0.3, shuffle=False)

# Step 2: Split the remaining 30% into test and validation (14% and 15%)
test_valid_split = train_test_split['test'].train_test_split(test_size=0.5, shuffle=False)

# Step 3: Combine into a new DatasetDict with train, test, and validation sets
split_dataset = DatasetDict({
    'train': train_test_split['train'],         # 70% of the data
    'test': test_valid_split['train'],          # 15% of the data
    'validation': test_valid_split['test']      # 15% of the data
})
split_dataset

In [ ]:
new_path = 'YOUR/OWN/PATH'  # Specify your own path here
split_dataset.save_to_disk(new_path)